In [4]:
# =========================
# IMPORTS
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

from lightgbm import LGBMClassifier


# Find dataset path automatically
base_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    if 'train.csv' in filenames:
        base_path = dirname
        break

print("Dataset path:", base_path)

train = pd.read_csv(f'{base_path}/train.csv')
test = pd.read_csv(f'{base_path}/test.csv')



# =========================
# PREPROCESSING
# =========================
X = train.drop(['id', 'Irrigation_Need'], axis=1)
y = train['Irrigation_Need']

# Encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# One-hot encoding
X = pd.get_dummies(X)
test_features = pd.get_dummies(test.drop(['id'], axis=1))

# Align columns
X, test_features = X.align(test_features, join='left', axis=1, fill_value=0)

# =========================
# CROSS VALIDATION + TRAINING
# =========================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

test_preds = np.zeros((test_features.shape[0], len(le.classes_)))
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_encoded)):
    print(f"Fold {fold+1}")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    model = LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='multi_logloss',
        callbacks=[]
    )
    
    # Validation prediction
    val_preds = model.predict(X_val)
    score = balanced_accuracy_score(y_val, val_preds)
    cv_scores.append(score)
    
    print(f"Balanced Accuracy: {score}")
    
    # Test prediction (probabilities for averaging)
    test_preds += model.predict_proba(test_features) / skf.n_splits

# =========================
# FINAL RESULTS
# =========================
print("\nMean CV Score:", np.mean(cv_scores))

# Convert predictions
final_preds = np.argmax(test_preds, axis=1)
final_preds = le.inverse_transform(final_preds)

# =========================
# SUBMISSION FILE
# =========================
submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': final_preds
})

submission.to_csv('submission.csv', index=False)

print("Submission file created!")

Dataset path: /kaggle/input/competitions/playground-series-s6e4
Fold 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.059813 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2728
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 43
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
Balanced Accuracy: 0.9671677506541819
Fold 2
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015016 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2729
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 43
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start 